# novel-ocr : 소설 스캔 이미지 → 텍스트 추출

Colab에서 실행하는 노트북입니다. 위에서부터 순서대로 셀을 실행하세요 (Shift+Enter).
OCR 엔진은 **PaddleOCR**(한글 인쇄체 받침 구분이 EasyOCR보다 정확)을 사용합니다.

**사용 순서**
1. 상단 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU** 선택
2. 1~3번 셀 실행 (설치, GPU 확인, Drive 연결)
3. 4번 셀에서 `IMAGE_SOURCE`(zip 파일 또는 이미지 폴더 경로)만 수정
4. 5~8번 셀 실행 → **8번에서 한 장 테스트 결과를 먼저 확인**
5. 9번 셀로 전체 처리 (끊겨도 다시 실행하면 이어서 처리)
6. 10번 셀로 폴더별 페이지를 한 문서로 합치기

## 1. 라이브러리 설치

PaddlePaddle(GPU)과 PaddleOCR 3.x를 설치합니다. GPU 버전은 Colab의 Python/CUDA 버전에 따라 받을 수 있는
저장소가 달라서, **여러 곳을 차례로 시도하고 모두 실패하면 자동으로 CPU 버전을 설치**합니다.

- 설치 진행 상황(내려받는 파일, 경과 시간)이 실시간으로 출력됩니다. PaddlePaddle GPU는 용량이 커서 몇 분 걸려요.
- 설치가 끝나면 **반드시 상단 메뉴 런타임 > 세션 다시 시작(Restart session)**을 한 번 해주세요. 재시작 후에는 이 셀은 건너뛰고 2번부터 실행하면 됩니다.
- 마지막 줄에 `설치 결과: CPU`가 나오면 동작은 하지만 느립니다. 그 경우 이 셀의 출력을 그대로 복사해서 알려주세요.

In [ ]:
import subprocess, sys, threading, time

print('Python', sys.version.split()[0], flush=True)

# pip 출력 중 진행 상황을 알 수 있는 줄만 보여줌 (전체를 다 찍으면 너무 길어서)
_SHOW = ('Collecting', 'Downloading', 'Installing collected', 'Successfully', 'ERROR', 'Found existing', 'Uninstalling')


def run_pip(title, *args):
    # pip를 실행하면서 진행 상황을 실시간으로 출력. 15초 넘게 새 출력이 없으면 '진행 중' 표시.
    print(f'\n==> {title}', flush=True)
    start = time.time()
    last = [start]
    tail = []
    done = threading.Event()

    def heartbeat():
        while not done.wait(10):
            if time.time() - last[0] >= 15:
                print(f'    ... 진행 중 ({int(time.time() - start)}초 경과, 큰 파일을 받는 중일 수 있어요)', flush=True)

    threading.Thread(target=heartbeat, daemon=True).start()
    proc = subprocess.Popen([sys.executable, '-m', 'pip', *args],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        line = line.rstrip()
        if not line:
            continue
        last[0] = time.time()
        tail = (tail + [line])[-3:]
        if line.lstrip().startswith(_SHOW):
            print('    ' + line.strip()[:160], flush=True)
    proc.wait()
    done.set()
    ok = proc.returncode == 0
    print(f'    -> {"성공" if ok else "실패"} ({int(time.time() - start)}초)', flush=True)
    if not ok:
        print('    마지막 메시지:', tail[-1] if tail else '(없음)', flush=True)
    return ok


PADDLE_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable'
GPU_CANDIDATES = [
    ('PaddlePaddle GPU (CUDA 12.6 저장소)', ['paddlepaddle-gpu', '--extra-index-url', f'{PADDLE_INDEX}/cu126/']),
    ('PaddlePaddle GPU (CUDA 11.8 저장소)', ['paddlepaddle-gpu', '--extra-index-url', f'{PADDLE_INDEX}/cu118/']),
    ('PaddlePaddle GPU (PyPI)', ['paddlepaddle-gpu']),
]

installed = None
for i, (title, args) in enumerate(GPU_CANDIDATES, 1):
    if run_pip(f'[1/4] {title} 설치 시도 ({i}/{len(GPU_CANDIDATES)}) - 용량이 커서 몇 분 걸려요', 'install', *args):
        installed = 'GPU'
        break

if installed is None:
    print('\nGPU 버전 설치가 모두 실패해서 CPU 버전으로 설치합니다.', flush=True)
    installed = 'CPU' if run_pip('[1/4] PaddlePaddle CPU 설치', 'install', 'paddlepaddle') else None

run_pip('[2/4] PaddleOCR 설치', 'install', 'paddleocr>=3.0,<4')
run_pip('[3/4] Kiwi (한국어 띄어쓰기 교정) 설치', 'install', 'kiwipiepy')

# Colab 기본 PyTorch 제거: Paddle GPU가 설치하면서 바꾼 CUDA 통신 라이브러리(NCCL)와 충돌해
# paddleocr을 불러올 때 torch 로딩 에러(undefined symbol: ncclCommShrink)가 납니다. 이 노트북은 torch를 쓰지 않습니다.
run_pip('[4/4] Colab 기본 PyTorch 제거 (Paddle과 충돌 방지)', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio')

print('\n' + '=' * 50)
print('설치 결과:', installed or '실패 (위 출력을 복사해서 알려주세요)')
print('>> 이제 상단 메뉴 [런타임 > 세션 다시 시작]을 누른 뒤 2번 셀부터 실행하세요.')

## 2. GPU 확인
`GPU 사용: True`가 나와야 합니다. `False`면 런타임 유형이 GPU가 아니거나 CPU 버전 Paddle이 설치된 것입니다 (CPU로도 동작은 함).

In [ ]:
import paddle
_gpu = paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
print('Paddle 버전:', paddle.__version__)
print('GPU 사용:', _gpu)

## 3. Google Drive 연동
실행하면 인증 창이 뜹니다. 로그인 후 권한을 허용하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 설정

`IMAGE_SOURCE`에 **zip 파일 경로** 또는 **이미지 폴더 경로**를 적으세요.
경로는 `/content/drive/MyDrive/` 뒤에 "내 드라이브"부터의 폴더/파일 이름을 그대로 이어 적으면 됩니다.

예) 내 드라이브 > 소설 > 1권.zip → `'/content/drive/MyDrive/소설/1권.zip'`

파일 이름이 헷갈리면 이 셀 아래의 `os.listdir(...)` 주석을 풀어서 폴더 내용을 확인하세요.

In [ ]:
import os

IMAGE_SOURCE  = '/content/drive/MyDrive/소설/파일명.zip'      # zip 파일 또는 이미지 폴더
OUTPUT_FOLDER = '/content/drive/MyDrive/novel_ocr_output'     # 결과 저장 폴더 (Drive)
TEXT_MIRROR_FOLDER = os.path.join(OUTPUT_FOLDER, 'texts')     # 원본과 같은 폴더 구조로 페이지별 txt 생성

EXTRACT_FOLDER = '/content/extracted_novel'  # 압축을 풀어둘 임시 위치 (Colab 임시 디스크)
HANDLE_HANJA = True    # 괄호 안 한자 보정 (예: 진동왕(鎭東王)). 끄려면 False

# print(os.listdir('/content/drive/MyDrive/소설'))   # 폴더 안 파일 이름 확인용

os.makedirs(TEXT_MIRROR_FOLDER, exist_ok=True)
IMAGE_FOLDER = IMAGE_SOURCE   # 5번 셀에서 실제 작업 폴더로 바뀝니다
print('원본:', IMAGE_SOURCE, '| 존재:', os.path.exists(IMAGE_SOURCE))
print('결과 폴더:', TEXT_MIRROR_FOLDER)

## 5. 압축 풀기 / 작업 폴더 준비 (항상 실행)

- zip이면 풀고, 풀린 안에 zip이 또 있으면 **끝까지 반복해서** 풉니다 (zip 속 zip 대응).
- 윈도우에서 만든 zip의 **한글 파일명 깨짐(cp949)**을 자동 보정합니다.
- 맥에서 만든 zip의 `__MACOSX`, `._파일` 같은 찌꺼기는 제외합니다.
- 이미지 폴더를 지정한 경우에도 Colab 임시 디스크로 복사해서 처리 속도를 높입니다.
- Colab 임시 디스크는 세션이 끊기면 사라지므로, **다음날 이어서 할 때도 이 셀은 다시 실행**해야 합니다 (OCR 결과는 Drive에 있어서 안전).

In [ ]:
import re
import shutil
import zipfile

CLEAN_EXTRACT = True   # 실행할 때마다 임시 폴더를 비우고 새로 풂 (다른 소설 파일이 섞이지 않게)

_UNSUPPORTED_ARCHIVE_RE = re.compile(r'\.(7z|rar|alz|egg|tar|gz|z\d{2}|zip\.\d{3}|\d{3})$', re.IGNORECASE)


def _decode_zip_name(info):
    """UTF-8 플래그가 없는 zip(윈도우 한글 zip 등)의 파일명을 올바르게 복원."""
    if info.flag_bits & 0x800:
        return info.filename
    try:
        raw = info.filename.encode('cp437')
    except UnicodeEncodeError:
        return info.filename
    for enc in ('utf-8', 'cp949'):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return info.filename


def safe_extract(zip_path, dest):
    """한글 파일명 보정 + 맥 찌꺼기 제외 + 경로 탈출(../) 방지하며 압축 해제."""
    dest = os.path.abspath(dest)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = _decode_zip_name(info).replace('\\', '/')
            parts = [p for p in name.split('/') if p not in ('', '.')]
            if not parts or '..' in parts or '__MACOSX' in parts or parts[-1].startswith('._'):
                continue
            target = os.path.join(dest, *parts)
            if name.endswith('/'):
                os.makedirs(target, exist_ok=True)
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with zf.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)


def extract_all_nested_zips(root):
    """root 아래의 zip을 모두 풀고, 풀린 결과에 zip이 또 있으면 없어질 때까지 반복."""
    failed = []
    while True:
        zips = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs if f.lower().endswith('.zip')]
        if not zips:
            break
        for zp in zips:
            rel = os.path.relpath(zp, root)
            try:
                safe_extract(zp, zp[:-4])          # 파일명.zip -> 파일명/ 폴더에 풀기
                print('풀림:', rel)
            except Exception as e:                 # 손상/암호/비표준 zip 등
                failed.append((rel, f'{type(e).__name__}: {e}'))
                print('[경고] 풀기 실패, 건너뜀:', rel, '-', e)
            finally:
                os.remove(zp)                      # 임시 폴더 안의 사본만 지움 (Drive 원본은 그대로)
    return failed


def prepare_work_folder(source, work):
    if not os.path.exists(source):
        raise FileNotFoundError(f'경로를 찾을 수 없습니다: {source}  (4번 셀의 IMAGE_SOURCE 확인)')
    same = os.path.abspath(source) == os.path.abspath(work)
    if CLEAN_EXTRACT and not same and os.path.exists(work):
        shutil.rmtree(work)
    os.makedirs(work, exist_ok=True)

    if os.path.isfile(source):
        if not source.lower().endswith('.zip'):
            raise ValueError(f'zip 파일이나 폴더만 지원합니다: {source}')
        safe_extract(source, work)
        print('최초 zip 풀기 완료 ->', work)
    elif not same:
        shutil.copytree(source, work, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('__MACOSX', '._*'))
        print('폴더 복사 완료 ->', work)

    failed = extract_all_nested_zips(work)

    leftovers = [os.path.relpath(os.path.join(d, f), work)
                 for d, _, fs in os.walk(work) for f in fs if _UNSUPPORTED_ARCHIVE_RE.search(f)]
    if leftovers:
        print('\n[경고] 이 노트북이 풀 수 없는 압축(분할압축/7z/rar/alz/egg 등)이 있습니다:')
        for x in leftovers[:20]:
            print('   ', x)
    if failed:
        print(f'\n[경고] 풀지 못한 zip {len(failed)}개:')
        for rel, err in failed[:20]:
            print('   ', rel, '-', err)
    return work


IMAGE_FOLDER = prepare_work_folder(IMAGE_SOURCE, EXTRACT_FOLDER)
print('\n작업 폴더 준비 완료:', IMAGE_FOLDER)

## 6. OCR 엔진 준비 (PaddleOCR)
처음 실행 시 모델을 다운로드하므로 1~3분 걸릴 수 있습니다.
PaddleOCR 3.x/2.x 어느 쪽이 설치되어 있어도 동작하도록 맞춰져 있습니다.

In [ ]:
import numpy as np
from PIL import Image
import paddle
import paddleocr

PADDLEOCR_MAJOR = int(paddleocr.__version__.split('.')[0])
USE_GPU = paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
DEVICE = 'gpu:0' if USE_GPU else 'cpu'
print(f'PaddleOCR {paddleocr.__version__} / 장치: {DEVICE}')


def load_image_bgr(path):
    """jpg/png/gif(첫 프레임)/webp/bmp/tif 모두 같은 방식으로 읽기."""
    with Image.open(path) as im:
        im.seek(0)
        rgb = np.array(im.convert('RGB'))
    return np.ascontiguousarray(rgb[:, :, ::-1])   # PaddleOCR은 BGR 배열을 기대


if PADDLEOCR_MAJOR >= 3:
    from paddleocr import PaddleOCR, TextRecognition

    _ocr_ko = PaddleOCR(
        lang='korean', device=DEVICE,
        use_doc_orientation_classify=False,   # 스캔본은 방향/왜곡 보정 불필요 (속도↑, 왜곡 방지)
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    def ocr_lines(img_bgr):
        """-> [(박스 꼭짓점 4개, 텍스트, 점수), ...]"""
        res = _ocr_ko.predict(img_bgr)
        if not res:
            return []
        r = res[0]
        return [(np.asarray(p).reshape(-1, 2).tolist(), t, float(s))
                for p, t, s in zip(r['rec_polys'], r['rec_texts'], r['rec_scores'])]

    _rec_han = None
    if HANDLE_HANJA:
        for _name in ('PP-OCRv5_server_rec', 'chinese_cht_PP-OCRv3_mobile_rec'):
            try:
                _rec_han = TextRecognition(model_name=_name, device=DEVICE)
                print('한자 인식 모델:', _name)
                break
            except Exception as e:
                print(f'[경고] {_name} 로드 실패: {e}')

    def recognize_hanja(crop_bgr):
        if _rec_han is None:
            return ''
        out = _rec_han.predict(crop_bgr, batch_size=1)
        return out[0]['rec_text'] if out else ''

else:  # PaddleOCR 2.x
    from paddleocr import PaddleOCR

    _ocr_ko = PaddleOCR(lang='korean', use_angle_cls=False, use_gpu=USE_GPU, show_log=False)
    _rec_han = PaddleOCR(lang='chinese_cht', use_angle_cls=False, use_gpu=USE_GPU,
                         show_log=False) if HANDLE_HANJA else None

    def ocr_lines(img_bgr):
        res = _ocr_ko.ocr(img_bgr, cls=False)
        lines = res[0] if res and res[0] else []
        return [(box, text, float(score)) for box, (text, score) in lines]

    def recognize_hanja(crop_bgr):
        if _rec_han is None:
            return ''
        res = _rec_han.ocr(crop_bgr, det=False, cls=False)   # 인식만 (영역 검출 X)
        return res[0][0][0] if res and res[0] else ''

print('엔진 준비 완료')

## 7. 처리 함수 (줄 정렬 · 문단 · 한자 보정 · 파일 찾기)

- **줄 정렬**: 같은 높이에 있는 글자 영역들을 한 줄로 묶고 왼쪽→오른쪽 순서로 합칩니다 (단어 순서 뒤섞임 방지).
- **문단**: 들여쓰기나 넓은 줄 간격이 있으면 빈 줄로 문단을 구분합니다.
- **한자 보정** (`HANDLE_HANJA=True`일 때): 한글 모델은 한자를 모르기 때문에 `진동왕(鎭東王)`의 괄호 안이 엉뚱한 글자로 나옵니다.
  괄호가 있는 줄만 한자 모델로 한 번 더 읽어서, **괄호 순서대로** 안쪽 내용을 한자로 바꿔 끼웁니다.
  괄호 개수가 안 맞으면 글자 위치를 비율로 계산해 괄호 부분만 잘라서 다시 읽습니다.
- 줄바꿈은 책의 줄 그대로 유지합니다 (한국어 조판은 단어 중간에서도 줄이 바뀌어서, 자동으로 이어 붙이면 띄어쓰기가 틀어질 수 있음).

In [ ]:
import re

_CJK_RE = re.compile(r'[㐀-䶿一-鿿豈-﫿]')
_PAREN_RE = re.compile(r'[(（]([^()（）]*)[)）]')


def _xyxy(poly):
    xs = [p[0] for p in poly]
    ys = [p[1] for p in poly]
    return min(xs), min(ys), max(xs), max(ys)


def _pad_white(crop, px=8):
    return np.pad(crop, ((px, px), (px, px), (0, 0)), mode='constant', constant_values=255)


def _crop(img, x1, y1, x2, y2):
    H, W = img.shape[:2]
    x1, x2 = int(max(0, x1)), int(min(W, x2))
    y1, y2 = int(max(0, y1)), int(min(H, y2))
    if x2 - x1 < 4 or y2 - y1 < 4:
        return None
    return _pad_white(img[y1:y2, x1:x2])


def _char_units(ch):
    """글자 폭 가중치: 한글/한자/전각 = 1, 공백/영문/반각 기호 = 0.5"""
    return 0.5 if (ch.isspace() or ord(ch) < 0x1100) else 1.0


def _crop_char_range(img, box, text, s, e, pad_units=0.4):
    x1, y1, x2, y2 = box
    units = [_char_units(c) for c in text]
    per = (x2 - x1) / (sum(units) or 1)
    h = y2 - y1
    return _crop(img,
                 x1 + (sum(units[:s]) - pad_units) * per, y1 - 0.1 * h,
                 x1 + (sum(units[:e]) + pad_units) * per, y2 + 0.1 * h)


def _only_cjk(s):
    return ''.join(_CJK_RE.findall(s))


def _is_all_cjk(s):
    s = s.replace(' ', '')
    return bool(s) and all(_CJK_RE.match(c) for c in s)


def _hangul_run_before(text, pos):
    """괄호 바로 앞 한글 단어의 음절 수. 한자 병기는 보통 한자 1자 = 한글 1음절 (진동왕 -> 鎭東王)."""
    n = 0
    while pos - n - 1 >= 0 and '가' <= text[pos - n - 1] <= '힣':
        n += 1
    return n


def _choose_hanja(korean_inner, han_cjk, expected_n):
    """한자 모델 결과를 쓸지 결정. None이면 원래 텍스트 유지."""
    k = _only_cjk(korean_inner)
    if not han_cjk or han_cjk == k:
        return None
    if expected_n:
        if len(k) == expected_n:            # 한글 모델이 이미 글자 수만큼 한자를 읽음
            return None
        if len(k) < len(han_cjk) <= expected_n:
            return han_cjk
        return None
    # 앞에 한글 단어가 없는 괄호: 한글 모델이 한자를 일부라도 읽은 경우에만 보강
    return han_cjk if k and len(han_cjk) > len(k) else None


def fix_hanja_in_box(img, poly, text):
    groups = list(_PAREN_RE.finditer(text))
    if not groups:
        return text
    box = _xyxy(poly)
    h = box[3] - box[1]
    expected = {m: _hangul_run_before(text, m.start(0)) for m in groups}
    repl = {}

    # 1) 줄 전체를 한자 모델로 읽고, 괄호 순서대로 짝지어 안쪽만 비교
    line_crop = _crop(img, box[0], box[1] - 0.1 * h, box[2], box[3] + 0.1 * h)
    han_groups = list(_PAREN_RE.finditer(recognize_hanja(line_crop))) if line_crop is not None else []
    if len(han_groups) == len(groups):
        for mk, mh in zip(groups, han_groups):
            c = _choose_hanja(mk.group(1), _only_cjk(mh.group(1)), expected[mk])
            if c:
                repl[(mk.start(1), mk.end(1))] = c

    # 2) 아직 글자 수가 모자란 괄호는 그 부분만 넉넉하게 잘라서 다시 읽기
    for m in groups:
        key = (m.start(1), m.end(1))
        cur = repl.get(key, m.group(1))
        n_exp = expected[m]
        if n_exp and len(_only_cjk(cur)) == n_exp:
            continue
        # 한글 모델이 한자를 빼먹었으면 괄호 안 폭을 한글 음절 수만큼 있다고 보고 위치 계산
        inner_len = max(len(m.group(1)), n_exp)
        virt = text[:m.start(1)] + '一' * inner_len + text[m.end(1):]
        crop = _crop_char_range(img, box, virt, m.start(0), m.start(1) + inner_len + 1, pad_units=1.5)
        if crop is None:
            continue
        t = recognize_hanja(crop)
        mh = _PAREN_RE.search(t)
        c = _choose_hanja(cur, _only_cjk(mh.group(1) if mh else t), n_exp)
        if c:
            repl[key] = c

    for s_, e_ in sorted(repl, reverse=True):
        text = text[:s_] + repl[(s_, e_)] + text[e_:]
    return text


def assemble_text(items, img=None, handle_hanja=False):
    """items: [(poly, text, score)] -> 줄/문단이 정리된 텍스트"""
    boxes = []
    for poly, text, _score in items:
        text = (text or '').strip()
        if not text:
            continue
        if handle_hanja and img is not None and ('(' in text or '（' in text):
            text = fix_hanja_in_box(img, poly, text)
        x1, y1, x2, y2 = _xyxy(poly)
        boxes.append({'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'text': text})
    if not boxes:
        return ''

    # 같은 높이의 박스를 한 줄로 묶기
    boxes.sort(key=lambda b: (b['y1'] + b['y2']) / 2)
    rows = []
    for b in boxes:
        if rows:
            r = rows[-1]
            overlap = min(r['y2'], b['y2']) - max(r['y1'], b['y1'])
            if overlap > 0.5 * min(b['y2'] - b['y1'], r['y2'] - r['y1']):
                r['items'].append(b)
                r['y1'], r['y2'] = min(r['y1'], b['y1']), max(r['y2'], b['y2'])
                continue
        rows.append({'y1': b['y1'], 'y2': b['y2'], 'items': [b]})
    for r in rows:
        r['items'].sort(key=lambda b: b['x1'])
        r['x1'] = r['items'][0]['x1']
        r['text'] = ' '.join(b['text'] for b in r['items'])

    # 문단 구분: 들여쓰기 / 평소보다 넓은 줄 간격 / 앞줄이 오른쪽 끝까지 안 찬 경우(문단 마지막 줄)
    med_h = sorted(r['y2'] - r['y1'] for r in rows)[len(rows) // 2]
    left = min(r['x1'] for r in rows)
    right = max(r['items'][-1]['x2'] for r in rows)
    gaps = sorted(rows[i + 1]['y1'] - rows[i]['y2'] for i in range(len(rows) - 1))
    med_gap = gaps[len(gaps) // 2] if gaps else 0

    def indented(r):
        return r['x1'] - left > 0.3 * med_h

    # 페이지 첫 줄이 들여쓰기면 맨 앞에 공백 하나를 둬서 '새 문단으로 시작'을 표시 (합치기 단계에서 사용)
    out = [(' ' if len(rows) > 1 and indented(rows[0]) else '') + rows[0]['text']]
    for prev, cur in zip(rows, rows[1:]):
        wide_gap = (cur['y1'] - prev['y2']) > med_gap + 0.35 * med_h
        short_prev = right - prev['items'][-1]['x2'] > 1.5 * med_h
        out.append('\n\n' if (indented(cur) or wide_gap or short_prev) else '\n')
        out.append(cur['text'])
    return ''.join(out)


def ocr_image(path):
    img = load_image_bgr(path)
    return assemble_text(ocr_lines(img), img, handle_hanja=HANDLE_HANJA)


IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tif', '.tiff'}


def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]


def list_images(root):
    """하위 폴더까지 이미지 찾기 (확장자 대소문자 무관, 맥 찌꺼기 제외, 사람 기준 숫자 순서)"""
    found = []
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x != '__MACOSX']
        for f in files:
            if not f.startswith('._') and os.path.splitext(f)[1].lower() in IMAGE_EXTS:
                found.append(os.path.join(d, f))
    return sorted(found, key=lambda p: natural_key(os.path.relpath(p, root)))


def build_output_map(root, images, out_root):
    """이미지 -> (상대경로, 결과 txt 경로). 같은 이름(01.jpg, 01.png)이 겹치면 01.jpg.txt 식으로 구분."""
    stems = {}
    for p in images:
        stems.setdefault(os.path.splitext(os.path.relpath(p, root))[0], []).append(p)
    mapping = {}
    for p in images:
        rel = os.path.relpath(p, root)
        stem = os.path.splitext(rel)[0]
        txt_rel = (rel if len(stems[stem]) > 1 else stem) + '.txt'
        mapping[p] = (rel, os.path.join(out_root, txt_rel))
    return mapping

## 8. 한 장 먼저 테스트 (권장)

전체를 돌리기 전에 한 장만 처리해서 결과를 확인합니다. `TEST_IMAGE`를 비워두면 첫 번째 이미지를 씁니다.
결과가 이상하면 이 출력을 그대로 복사해서 물어보세요.

In [ ]:
import time

TEST_IMAGE = ''   # 특정 파일로 테스트하려면 전체 경로 입력

_path = TEST_IMAGE or list_images(IMAGE_FOLDER)[0]
print('테스트 이미지:', _path)
_t = time.time()
print('-' * 40)
print(ocr_image(_path))
print('-' * 40)
print(f'소요 시간: {time.time() - _t:.1f}초 (첫 실행은 모델 준비 때문에 더 느림)')

## 9. 전체 일괄 처리 (원본 폴더 구조 유지 · 이어하기 지원)

- 이미지 하나당 txt 하나를 `TEXT_MIRROR_FOLDER` 아래 **같은 하위 폴더 경로**로 저장합니다. 예: `1권/001.jpg` → `1권/001.txt`
- 이미 txt가 있는 이미지는 건너뛰므로, 세션이 끊겨도 **다시 실행하면 남은 것만** 처리합니다.
- 파일은 임시 이름으로 쓴 뒤 완성되면 이름을 바꾸므로, 중간에 끊겨도 반쪽짜리 txt가 "완료"로 취급되지 않습니다.
- 에러 난 이미지는 건너뛰고 `errors.log`에 기록합니다 (다음 실행 때 다시 시도).
- 확장자 대소문자 무관 (`.JPG`도 인식), 파일 순서는 사람 기준 숫자 순서(1, 2, …, 10).

In [ ]:
import re
from tqdm.auto import tqdm

def run_batch():
    if not os.path.isdir(IMAGE_FOLDER):
        raise RuntimeError('작업 폴더가 없습니다. 5번 셀(압축 풀기/작업 폴더 준비)을 먼저 실행하세요.')
    images = list_images(IMAGE_FOLDER)
    mapping = build_output_map(IMAGE_FOLDER, images, TEXT_MIRROR_FOLDER)
    todo = [p for p in images if not os.path.exists(mapping[p][1])]
    print(f'전체 {len(images)}장 / 이미 완료 {len(images) - len(todo)}장 / 이번에 처리 {len(todo)}장')

    error_log = os.path.join(OUTPUT_FOLDER, 'errors.log')
    n_err = 0
    for p in tqdm(todo, unit='장'):
        rel, out_path = mapping[p]
        try:
            text = ocr_image(p)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            tmp = out_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                f.write(text)
            os.replace(tmp, out_path)          # 다 쓴 뒤에만 완료 파일로 확정
        except Exception as e:
            n_err += 1
            with open(error_log, 'a', encoding='utf-8') as f:
                f.write(f'{rel}\t{type(e).__name__}: {e}\n')
            tqdm.write(f'[오류] {rel}: {e}')

    print(f'\n완료! 오류 {n_err}건' + (f' (자세한 내용: {error_log})' if n_err else ''))
    print('결과 폴더:', TEXT_MIRROR_FOLDER)


run_batch()

## 10. 폴더별로 한 문서로 합치기 (자연스러운 문단으로)

txt가 들어있는 **가장 안쪽 폴더마다**, 페이지 txt들을 숫자 순서(1, 2, …, 10)대로 이어 붙여 **문서 하나**로 저장합니다.

예) `texts/1권/1화/001.txt, 002.txt, …` → `merged/1권/1화.txt`

**책의 줄바꿈을 없애고 문단 단위로 이어 붙입니다** (`JOIN_LINES = True`).
- 같은 문단 안의 줄은 한 줄로 합칩니다. 예) `사람` + `을 수용할` → `사람을 수용할`
- 페이지가 문단 중간에서 끊겨도 다음 페이지와 자연스럽게 이어 붙입니다.
- 줄이 합쳐지는 자리와 OCR이 빠뜨린 띄어쓰기는 **Kiwi(한국어 띄어쓰기 교정)**로 채웁니다. 예) `가자리` → `가 자리`
- 문단 사이는 빈 줄 하나(`PARAGRAPH_SEPARATOR = '\n\n'`). 빈 줄 없이 한 줄에 한 문단을 원하면 `'\n'`.
- 책 줄바꿈을 그대로 두고 싶으면 `JOIN_LINES = False`.

원본 페이지별 txt는 그대로 두고 `OUTPUT_FOLDER/merged/`에 따로 만듭니다. 몇 번을 실행해도 새로 덮어씁니다.

In [ ]:
import re

MERGED_FOLDER = os.path.join(OUTPUT_FOLDER, 'merged')
JOIN_LINES = True              # True: 문단 단위로 이어 붙이기 / False: 책 줄바꿈 그대로
PARAGRAPH_SEPARATOR = '\n\n'   # 문단 사이 ('\n'이면 빈 줄 없이)
FIX_SPACING = True             # Kiwi로 띄어쓰기 교정 (줄 합친 자리 + OCR이 빠뜨린 띄어쓰기)

_spacer = None
if JOIN_LINES and FIX_SPACING:
    try:
        from kiwipiepy import Kiwi
        _kiwi = Kiwi()
        _spacer = lambda t: _kiwi.space(t, reset_whitespace=False)   # 있는 띄어쓰기는 두고, 빠진 곳만 채움
        print('띄어쓰기 교정: 사용')
    except Exception as e:
        print('[경고] kiwipiepy를 불러오지 못해 띄어쓰기 교정 없이 합칩니다:', e)

_CLOSE_PUNCT = tuple('.,!?…:;"\'”’」』》〉)]')


def _join_pair(a, b):
    """줄/페이지를 이어 붙이는 자리: 문장부호로 끝나면 한 칸 띄우고, 아니면 붙인 뒤 띄어쓰기 교정에 맡김."""
    a, b = a.rstrip(), b.lstrip()
    if not a or not b:
        return a or b
    return a + (' ' if a.endswith(_CLOSE_PUNCT) else '') + b


def _page_paragraphs(t):
    """페이지 txt -> (새 문단으로 시작하는가, [문단 텍스트, ...])"""
    starts_new = t[:1] in (' ', '　')
    paras = []
    for block in re.split(r'\n\s*\n', t.strip()):
        lines = [ln.strip() for ln in block.split('\n') if ln.strip()]
        if not lines:
            continue
        p = lines[0]
        for ln in lines[1:]:
            p = _join_pair(p, ln)
        paras.append(p)
    return starts_new, paras


def merge_pages(texts):
    if not JOIN_LINES:
        return '\n'.join(t.strip() for t in texts if t.strip()) + '\n'
    paras = []
    for t in texts:
        starts_new, ps = _page_paragraphs(t)
        if not ps:
            continue
        if paras and not starts_new:
            paras[-1] = _join_pair(paras[-1], ps[0])   # 앞 페이지 문단에 이어 붙임
            ps = ps[1:]
        paras.extend(ps)
    if _spacer:
        paras = [_spacer(p) for p in paras]
    return PARAGRAPH_SEPARATOR.join(paras) + '\n'


def merge_leaf_folders(text_root, merged_root):
    groups = {}
    for d, _, files in os.walk(text_root):
        txts = [f for f in files if f.endswith('.txt')]
        if txts:
            groups[d] = sorted(txts, key=natural_key)

    expected = {}
    if 'IMAGE_FOLDER' in globals() and os.path.isdir(IMAGE_FOLDER):
        imgs = list_images(IMAGE_FOLDER)
        for _rel, out in build_output_map(IMAGE_FOLDER, imgs, text_root).values():
            expected.setdefault(os.path.dirname(out), set()).add(os.path.basename(out))

    n_docs, n_warn = 0, 0
    for d in sorted(groups, key=lambda x: natural_key(os.path.relpath(x, text_root))):
        rel = os.path.relpath(d, text_root)
        out_path = os.path.join(merged_root, '전체.txt' if rel == '.' else rel + '.txt')
        texts = []
        for f in groups[d]:
            with open(os.path.join(d, f), encoding='utf-8') as fh:
                texts.append(fh.read())
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        tmp = out_path + '.tmp'
        with open(tmp, 'w', encoding='utf-8') as fh:
            fh.write(merge_pages(texts))
        os.replace(tmp, out_path)
        n_docs += 1

        missing = len(expected.get(d, set()) - set(groups[d]))
        note = f'   [경고] 아직 처리 안 된 페이지 {missing}장' if missing else ''
        n_warn += bool(missing)
        print(f'{os.path.relpath(out_path, merged_root)}  <- 페이지 {len(groups[d])}장{note}')

    for d in sorted(set(expected) - set(groups), key=lambda x: natural_key(os.path.relpath(x, text_root))):
        n_warn += 1
        print(f'[경고] {os.path.relpath(d, text_root)} : 처리된 페이지가 없어 건너뜀')

    print(f'\n문서 {n_docs}개 생성 -> {merged_root}' + (f'  (경고 {n_warn}건: 9번을 다시 실행한 뒤 이 셀을 다시 실행하세요)' if n_warn else ''))


merge_leaf_folders(TEXT_MIRROR_FOLDER, MERGED_FOLDER)

## 11. 결과 확인

In [ ]:
all_txt = sorted((os.path.join(d, f) for d, _, fs in os.walk(TEXT_MIRROR_FOLDER) for f in fs if f.endswith('.txt')),
                 key=lambda p: natural_key(os.path.relpath(p, TEXT_MIRROR_FOLDER)))
print(f'생성된 txt 파일: {len(all_txt)}개\n')
for p in all_txt[:3]:
    with open(p, encoding='utf-8') as f:
        print(f'--- {os.path.relpath(p, TEXT_MIRROR_FOLDER)} ---')
        print(f.read()[:300], '\n')

## 12. (작업 끝났을 때) Drive에 확실히 저장
Colab은 Drive 쓰기를 잠깐 모아뒀다가 반영합니다. 다 끝나고 창을 닫기 전에 실행해두면 안전합니다.

In [ ]:
drive.flush_and_unmount()
print('Drive 저장 완료. 다시 작업하려면 3번 셀부터 실행하세요.')